In [13]:
class DynamicOrder:
    def __init__(self, order_id):
        self.attributes = dict()
        self.methods = dict()
        
        self.attributes["order_id"] = order_id
        self.attributes["items"] = []

    # -------------------------------------------------------------
    def add_attribute(self, att_name, att_value):
        self.attributes[att_name] = att_value

    def edit_attribute(self, att_name, new_value):
        if att_name in self.attributes:
            self.attributes[att_name] = new_value
        else:
            print("attribute not found")

    def del_attribute(self, att_name):
        if att_name in self.attributes:
            del self.attributes[att_name]
        else:
            print("attribute not found")

    def get_attribute(self, att_name):
        return self.attributes.get(att_name)

     # -------------------------------------------------------------
    def add_method(self, name_method, method_func):
        self.methods[name_method] = method_func

    def call_method(self, name_method, *args, **kwargs):
        if name_method in self.methods:
            return self.methods[name_method](self, *args, **kwargs)
        else:
            print(f"Method '{name_method}' not found!")

    # -------------------------------------------------------------
    def process_order(self):
        print(f"\n--- Processing Dynamic Order #{self.attributes.get('order_id')} ---")
        
        payment_service = self.attributes.get("payment_strategy")
        shipping_service = self.attributes.get("shipping_provider")
        discount_service = self.attributes.get("discount_rule")
        
        if not payment_service or not shipping_service:
            print("Error: Missing Payment or Shipping components!")
            return False

        items = self.attributes.get("items", [])
        subtotal = sum(item["price"] for item in items)
        total_weight = sum(item["weight_kg"] for item in items)

        discounted_subtotal = discount_service.apply_discount(subtotal) if discount_service else subtotal
        shipping_cost = shipping_service.calculate_cost(total_weight)
        final_total = discounted_subtotal + shipping_cost

        print(f"Subtotal: ${subtotal} | After Discount: ${discounted_subtotal} | Shipping: ${shipping_cost} | Total: ${final_total}")

        if payment_service.pay(final_total):
            tracking = shipping_service.ship(self.attributes.get("order_id"))
            print(f"Success: {tracking}")
            return True
        return False

In [14]:
from abc import ABC, abstractmethod

class IShippingProvider(ABC):
    @abstractmethod
    def calculate_cost(self, weight_kg: float) -> float:
        pass

    @abstractmethod
    def ship(self, order_id: str) -> str:
        pass


class AramexShipping(IShippingProvider):
    def calculate_cost(self, weight_kg: float) -> float:
        return 50.0 + (weight_kg * 10.0)

    def ship(self, order_id: str) -> str:
        return f"[Aramex Express] Shipment waybill created for order '{order_id}'."


class BostaShipping(IShippingProvider):
    def calculate_cost(self, weight_kg: float) -> float:
        return 30.0 + (weight_kg * 8.0)

    def ship(self, order_id: str) -> str:
        return f"[Bosta Deliveries] Order '{order_id}' dispatched to local hub."

# ==========================================

class IPaymentStrategy(ABC):
   
    @abstractmethod
    def pay(self, amount: float) -> bool:
        pass

class VodafoneCashPayment(IPaymentStrategy):
 
    def __init__(self, phone_number: str):
        self.__phone_number = phone_number  

    def pay(self, amount: float) -> bool:
        print(f"--> Initiating Vodafone Cash transaction of {amount:.2f} EGP for {self.__phone_number}...")
        print("--> OTP Verified. Payment Successful!")
        return True


class CreditCardPayment(IPaymentStrategy):

    def __init__(self, card_number: str, holder_name: str):
        self.__card_number = card_number
        self.holder_name = holder_name

    def pay(self, amount: float) -> bool:
        masked_card = f"**** **** **** {self.__card_number[-4:]}"
        print(f"--> Charging {amount:.2f} EGP to Credit Card ({masked_card}) for {self.holder_name}...")
        print("--> Bank Response: APPROVED.")
        return True

# ==========================================

class IDiscountRule(ABC):

    @abstractmethod
    def apply_discount(self, total_amount: float) -> float:
        pass


class PromoCodeDiscount(IDiscountRule):

    def __init__(self, code: str, percentage: float):
        self.code = code
        self.percentage = percentage

    def apply_discount(self, total_amount: float) -> float:
        discount_value = total_amount * (self.percentage / 100.0)
        print(f"--> Applied Promo Code '{self.code}' ({self.percentage}% OFF: -{discount_value:.2f} EGP)")
        return total_amount - discount_value


class NoDiscount(IDiscountRule):

    def apply_discount(self, total_amount: float) -> float:
        return total_amount

In [15]:
# main.py

if __name__ == "__main__":

    my_order = DynamicOrder(order_id="ORD-2026-X")

    items = my_order.get_attribute("items")
    items.append({"name": "Mechanical Keyboard", "price": 1500.0, "weight_kg": 1.2})
    items.append({"name": "Wireless Mouse", "price": 500.0, "weight_kg": 0.3})

    print("=== [Scenario 1: Initial Setup] ===")
    my_order.add_attribute("shipping_provider", AramexShipping())
    my_order.add_attribute("payment_strategy", VodafoneCashPayment("01012345678"))
    my_order.add_attribute("discount_rule", PromoCodeDiscount("SAVE15", 15.0))

    my_order.process_order()

    print("\n=== [Scenario 2: Runtime Service Mutation] ===")

    my_order.edit_attribute("shipping_provider", BostaShipping())
    my_order.edit_attribute("payment_strategy", CreditCardPayment("4111222233334444", "Mohamed Hassan"))
    my_order.edit_attribute("discount_rule", NoDiscount())

    my_order.process_order()

=== [Scenario 1: Initial Setup] ===

--- Processing Dynamic Order #ORD-2026-X ---
--> Applied Promo Code 'SAVE15' (15.0% OFF: -300.00 EGP)
Subtotal: $2000.0 | After Discount: $1700.0 | Shipping: $65.0 | Total: $1765.0
--> Initiating Vodafone Cash transaction of 1765.00 EGP for 01012345678...
--> OTP Verified. Payment Successful!
Success: [Aramex Express] Shipment waybill created for order 'ORD-2026-X'.

=== [Scenario 2: Runtime Service Mutation] ===

--- Processing Dynamic Order #ORD-2026-X ---
Subtotal: $2000.0 | After Discount: $2000.0 | Shipping: $42.0 | Total: $2042.0
--> Charging 2042.00 EGP to Credit Card (**** **** **** 4444) for Mohamed Hassan...
--> Bank Response: APPROVED.
Success: [Bosta Deliveries] Order 'ORD-2026-X' dispatched to local hub.
